# Step 7 — Downstream Analysis: scWAT, vWAT, and Skeletal Muscle

## What this notebook is

This is where the biology happens. Steps 1–6 built the atlas. This step uses it to answer the paper's central question:

> **How does exercise training counteract the effects of obesity, and which cell types drive those changes across fat and muscle tissue?**

The answer, according to Yang et al. (2022): **mesenchymal stem cells (MSCs)** are the primary responders — more so than any other cell type, across all three tissues. Two gene programs dominate:
- **ECM remodeling** (Thbs1, Sparc): upregulated in MSCs by obesity, reversed by exercise training
- **Circadian rhythm** (Dbp, Tef, Nr1d2, Per3): upregulated in MSCs by exercise, suppressed in obesity

This was not known before this paper. Previous work focused on mature adipocytes and immune cells. The single-cell resolution revealed that the progenitor population — the stem cells — are the most responsive to both disease and intervention.

## Why single-cell resolution was necessary

Bulk RNA-seq (Step 8) found 1,386 DEGs across the three tissues. Those DEGs reflect the average transcriptional response across all cells. The problem: a gene that goes up in bulk could be increasing in *all* cell types, or it could be dramatically up in one rare cell type (like MSCs, which are ~5% of cells) while unchanged everywhere else. Without single-cell resolution, you cannot distinguish these cases, and you cannot identify the responsible cell type.

The scRNA-seq atlas answers: which specific cell types and cell states drive each DEG? The paper found that in vWAT, **>60% of cell-state DEGs were in ASCs** (the adipose MSC equivalent), even though ASCs are a minority of cells. Without single-cell resolution, the immune cell majority would have swamped this signal entirely.

## Study design

51 mice × 4 conditions × 3 tissues:

| Code | Diet | Exercise | Biological question |
|------|------|----------|---------------------|
| SC | Standard chow | Sedentary | Baseline (reference) |
| TC | Standard chow | Training | What does exercise do in healthy animals? |
| SH | High-fat diet | Sedentary | What does obesity do? |
| TH | High-fat diet | Training | Can exercise rescue obesity? (the key question) |

Three tissues: subcutaneous fat (scWAT), visceral fat (vWAT), skeletal muscle (SkM). Why all three? Because exercise effects are systemic — muscle signals to fat via secreted proteins (myokines), and fat signals back (adipokines). Profiling all three simultaneously allowed the paper to map **cross-tissue cell-cell communication** changes — a dimension of the exercise response invisible to single-tissue studies.

## Key results this notebook reproduces

- **Figure 3A**: The full 204,883-cell atlas tSNE colored by cell type — 22 cell types visible
- **Figure 3B/C**: Same atlas colored by tissue and intervention group
- Proportion changes: HFD increases macrophages and decreases mature adipocytes; exercise reverses this
- Cell-type-specific DEGs: 139 in scWAT, 502 in vWAT, 290 in SkM at cell-state level
- vWAT most strongly affected by HFD: 7.6× more DEGs than scWAT in the obesity comparison

## What this notebook does

1. Define the publication color palette for 22+ cell types and 4 experimental groups
2. Load phenotype metadata from Supplementary Table Z1 (42 samples → 39 after failures removed)
3. Document the study-specific parameter decisions for Steps 1–5 (with rationale)
4. Build the full atlas and 3 per-tissue integrated objects
5. Add metadata to all objects
6. Generate tSNE/UMAP feature plots colored by cell type, tissue, and condition

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from pathlib import Path

analysis_dir = Path(".")   # set to your project root

## Color palette

A consistent, publication-quality color scheme is defined once here and used throughout all plots. Colors are assigned at three levels:

- **Cell type** — 22 top-level cell types spanning adipogenic, immune, muscle, and stromal lineages. Colors are chosen to be perceptually distinct within each lineage group (e.g., shades of red/orange for adipogenic cells, blues for B/DC/immune lymphoid, greens for myeloid, pinks for muscle)
- **Tissue** — three colors for scWAT, vWAT, and SkM
- **Phenotype (pheno)** — four colors for the SC/TC/SH/TH experimental groups; these are the primary comparison groups for differential expression
- **Technical covariates** — tissue collection day (D1–D6) and time (morning/noon) are colored separately to check for technical confounding
- **Diet and exercise** — split-out versions of the phenotype for simpler two-group comparisons

In [ ]:
annotation_colors = {
    "cell_type": {
        # Adipogenic / stromal lineage
        "ASC":                    "#E41A1C",
        "FAP":                    "#CC2731",
        "FAP_Sca1-":              "#B53546",
        "Satellite":              "#9D425B",
        "Fibroblast":             "#CDA02C",
        "Tenocyte":               "#F87B0B",
        # Immune — lymphoid
        "B":                      "#4A72A6",
        "DC":                     "#3B87A3",
        "Neutrophil":             "#3E8E93",
        "ILC":                    "#419583",
        # Immune — myeloid
        "Macrophage":             "#449C72",
        "Mast":                   "#48A462",
        "Monocyte":               "#4BAB52",
        "NK":                     "#52A750",
        "Plasma":                 "#5D995D",
        "T":                      "#688B6A",
        # Muscle / structural
        "Muscle_Fiber":           "#E27699",
        "Smooth_Muscle":          "#F781BF",
        "Adipocyte":              "#F78184",
        "EC":                     "#C88DAC",
        "Pericyte":               "#C78DC8",
        "Epi":                    "#C88D8F",
        # Skeletal muscle myonuclei subtypes
        "Fast_myonuclei":         "#865070",
        "Slow_myonuclei":         "#DC6E37",
        "MTJ_myonuclei":          "#EA7521",
        "Unidentified_myonuclei": "#F87B0B",
        "NMJ_myonuclei":          "#FF8703",
        # Broad groupings
        "immune":                 "#475F49",
        "lymph":                  "#2E368F",
        "myeloid":                "#459C73",
    },
    "tissue": {
        "scWAT": "#D95F02",
        "vWAT":  "#7570B3",
        "SM":    "#1B9E77",
    },
    # SC = standard chow sedentary, SH = HFD sedentary
    # TC = standard chow training, TH = HFD training
    "pheno": {
        "SC": "#B9DBF4",
        "SH": "#F2B342",
        "TC": "#155289",
        "TH": "#E7872B",
    },
    "tissue_collection_day": {
        "D1": "#FBB4AE", "D2": "#B3CDE3", "D3": "#CCEBC5",
        "D4": "#DECBE4", "D5": "#FED9A6", "D6": "#FFFFCC",
    },
    "tissue_collection_time": {
        "morning": "#B3E2CD",
        "noon":    "#CBD5E8",
    },
    "diet": {
        "HFD":      "#E41A1C",
        "Standard": "#377EB8",
    },
    "exercise": {
        "training":  "#4DAF4A",
        "sedentary": "#984EA3",
    },
}

## Load phenotype metadata

Sample-level metadata is stored in `TableZ1.scRNA_sequencing_sample_metrics.xlsx` (Supplementary Table Z1 of the manuscript). This table records the library ID, pooled mouse ID, tissue collection day and time, diet, exercise group, and tissue for all 42 samples.

**`pooled_mouse_id`** encodes the experimental group as a prefix (`SC_`, `TC_`, `SH_`, `TH_`), which is extracted into a `pheno` column for clean group labeling in plots.

**Tissue collection day and time** are recorded because samples were dissected across multiple days and at different times of day. These are potential technical covariates — if all HFD samples happened to be dissected on D3 and all NCD samples on D1, apparent diet effects could be confounded with collection date. Tracking these variables allows post-hoc checks and covariate correction if needed.

Three libraries were removed due to quality issues identified in Step 3: D19-5431, D19-5443, and D19-5462. These are dropped from the metadata after loading.

In [ ]:
pheno_df = pd.read_excel(
    analysis_dir / "TableZ1.scRNA_sequencing_sample_metrics.xlsx",
    sheet_name=0,
    usecols="A:G",
    nrows=42,
)
pheno_df.columns = ["Library", "pooled_mouse_id", "tissue_collection_day",
                    "tissue_collection_time", "diet", "exercise", "tissue"]
pheno_df = pheno_df.set_index("Library")

# Standardize collection time labels
pheno_df["tissue_collection_time"] = pheno_df["tissue_collection_time"].map(
    lambda x: "morning" if x == "8am-9am" else "noon"
)

# Extract experimental group code from pooled_mouse_id prefix (e.g. "SC_Mouse1" -> "SC")
pheno_df["pheno"] = pheno_df["pooled_mouse_id"].str.replace("_.*", "", regex=True)
pheno_df["pheno"] = pd.Categorical(
    pheno_df["pheno"], categories=["SC", "TC", "SH", "TH"]
)

# Remove three failed libraries identified during Step 3 QC
failed_libs = ["D19-5431", "D19-5443", "D19-5462"]
pheno_df = pheno_df.drop(index=[l for l in failed_libs if l in pheno_df.index])

print(f"Samples retained: {len(pheno_df)}")
print(f"\nSamples per tissue:")
print(pheno_df["tissue"].value_counts().to_string())
print(f"\nSamples per pheno group:")
print(pheno_df["pheno"].value_counts().to_string())
pheno_df.head()

## Pipeline steps 1–4: processing and QC

The cells below call the individual pipeline notebooks (Steps 1–4) with the study-specific parameters used in the manuscript. These are documented here for reproducibility — the parameter choices reflect decisions made after inspecting the diagnostic plots from Steps 1 and 2.

**Key parameter decisions for this study:**

- `soupx_mode = "fixed_0.2"` — a fixed 20% ambient RNA contamination rate was used rather than the auto-estimate. This was chosen because these tissues (adipose and muscle) have high cell fragility during dissociation, leading to elevated ambient RNA that the auto-estimator sometimes underestimates.
- `per_mt_high = 30` — mitochondrial threshold set at 30% for per-sample filtering (higher than a typical 20% threshold). Adipose tissue has naturally higher mitochondrial content due to the metabolic activity of adipocytes and the abundance of mitochondria-rich beige adipocytes in exercised animals.
- `nfeature_low = 200, nfeature_high = 6000` — standard range covering the cell complexity expected in these tissues.
- `ncount_low = 500` — slightly higher than the typical 200 minimum to remove very shallow barcodes.
- `norm_mode = "sctransform"` for diagnostic plots (Step 2), `"regular"` for integration (Step 5). SCTransform is used for initial QC visualization where depth correction is important; regular normalization is used for integration because it is more computationally tractable at atlas scale.

In [ ]:
# These calls invoke the logic from the individual step notebooks.
# In a production run, import the helper functions from those notebooks
# or refactor them into a shared module.

# --- Step 1: CellRanger metrics ---
# cellranger_metrics(
#     cellranger_input_file="three_tissue_cellranger_input_file.txt",
#     target_folder=analysis_dir / "rdata"
# )

# --- Step 2: Diagnostic plots ---
# sample_level_diagnostic_plot(
#     cellranger_input_file="three_tissue_cellranger_input_file.txt",
#     species="mouse",
#     norm_mode="sctransform",
#     target_folder=analysis_dir / "plots"
# )

# --- Step 3: Per-sample processing ---
# sample_level_process(
#     cellranger_input_file="three_tissue_cellranger_input_file.txt",
#     species="mouse",
#     soupx_b=True,
#     soupx_mode="fixed_0.2",    # 20% fixed contamination rate
#     seurat_filt_b=True,
#     nfeature_low=200,
#     nfeature_high=6000,
#     ncount_low=500,
#     per_mt_high=30,             # permissive mt threshold for adipose/muscle
#     norm_mode="sctransform",
#     target_folder=analysis_dir / "rdata"
# )
# NOTE: Remove D19-5431, D19-5443, D19-5462 after this step (quality failures)

# --- Step 4: Pseudobulk clustering sanity check ---
# sample_level_pseudobulk_cluster(
#     input_file_directory=analysis_dir / "rdata" / "doubletFinder",
#     target_folder=analysis_dir / "plots",
#     pheno_df=pheno_df,
#     pheno_colors=annotation_colors
# )

print("Step 1-4 calls shown above as reference. Run the individual notebooks to execute them.")

## Step 5: Integration — full atlas and per-tissue objects

Integration is run twice:

1. **Full atlas** (`scwat_vwat_skm`) — all samples from all three tissues merged together. This gives a cross-tissue view of the cellular landscape and allows comparison of cell type proportions across tissues.

2. **Per-tissue objects** (scWAT, vWAT, SkM) — each tissue integrated separately. Per-tissue objects are used for within-tissue analyses: finding tissue-specific cell types, running differential expression within a tissue, and generating tissue-specific UMAPs where the resolution is not diluted by inter-tissue differences.

**Why `integration_method="merge"` rather than Harmony?**
The R script uses `merge` (simple concatenation) for both the atlas and per-tissue objects. This is a deliberate choice: samples within each tissue are from the same protocol and sequencing run structure, and the biological variation of interest (diet × exercise) should not be regressed out by batch correction. Using `merge` preserves the full between-condition variance that is the subject of downstream differential expression analysis. Batch correction would be appropriate if there were known systematic technical differences between batches, but the pseudobulk clustering in Step 4 confirmed that samples cluster primarily by tissue rather than by batch.

**`mt_cutoff = 10`** — a stricter mitochondrial threshold (10%) is applied at integration, tighter than the per-sample threshold of 30%. After merging, cells with high mt fractions that are outliers relative to the full atlas are removed. This two-stage approach catches cells that passed per-sample QC but look atypical in the broader context.

In [ ]:
# Paths to per-sample processed objects from Step 3
all_sample_files = [
    analysis_dir / "rdata" / "doubletFinder" / f"{lib}.h5ad"
    for lib in pheno_df.index
]

integration_params = dict(
    integration_method="merge",
    species="mouse",
    norm_mode="regular",
    rm_prolif=True,
    mt_cutoff=10,
    dims=50,
    res=0.4,
)

# Full atlas
# sample_integration(
#     input_file_list=all_sample_files,
#     df_name="scwat_vwat_skm",
#     target_folder=analysis_dir,
#     **integration_params
# )

# Per-tissue objects
# for tissue in pheno_df["tissue"].unique():
#     tissue_files = [
#         analysis_dir / "rdata" / "doubletFinder" / f"{lib}.h5ad"
#         for lib in pheno_df[pheno_df["tissue"] == tissue].index
#     ]
#     sample_integration(
#         input_file_list=tissue_files,
#         df_name=tissue,
#         target_folder=analysis_dir,
#         **integration_params
#     )

print("Integration calls shown above as reference. Run sample_integration.ipynb to execute.")

## Step 6: Add metadata to all objects

Phenotype metadata is joined onto all four objects (the full atlas and three per-tissue objects). After this step each cell knows its tissue, diet, exercise condition, collection day, and experimental group.

In [ ]:
df_names = ["scwat_vwat_skm", "scWAT", "vWAT", "SM"]

for df_name in df_names:
    obj_path = analysis_dir / "rdata" / f"{df_name}_combined.h5ad"
    if not obj_path.exists():
        print(f"Skipping {df_name} (file not found — run integration first)")
        continue

    adata = sc.read_h5ad(obj_path)

    # Left join: broadcast sample-level phenotype onto every cell
    existing = [c for c in pheno_df.columns if c in adata.obs.columns]
    if existing:
        adata.obs = adata.obs.drop(columns=existing)
    adata.obs = adata.obs.merge(
        pheno_df, left_on="sample_ID", right_index=True, how="left"
    )

    adata.write_h5ad(obj_path)
    print(f"{df_name}: {adata.n_obs:,} cells, metadata added, saved.")

## Feature plot utility

The `feature_plot` function in R renders any cell metadata or gene expression value onto a tSNE or UMAP embedding. It is the visualization workhorse for Step 7 — used to generate every colored UMAP/tSNE figure in the paper.

### Two modes

**Categorical** (e.g., cell type, tissue, pheno group): cells are colored by group membership using a fixed color dictionary. This is how cell type annotation figures are produced — assign cell type labels to clusters, then plot.

**Continuous** (e.g., gene expression, nCount_RNA): cells are colored on a gradient. The R code clips to the 1st and 9th decile (`min.cutoff="q1"`, `max.cutoff="q9"`) rather than the full range. This prevents a few extreme outlier cells from compressing the color scale so that most of the variation is invisible.

### Point size scaling

At 200,000+ cells, a standard point size of 1 produces a solid mass of overlapping dots. The function scales point size down automatically: 1.0 for < 100k cells, 0.1 for 100k–400k cells, 0.001 for > 400k cells. This keeps the embedding structure visible.

### Legend modes

- `"include"` — legend inside the plot
- `"none"` — no legend (for clean figures where labels are added in post-processing)
- `"separate"` — saves the legend as a standalone PDF, useful for figure assembly

In [ ]:
def feature_plot(adata, feature, var_class, reduction="umap",
                 feature_colors=None, target_folder=None,
                 file_format="png", label=False,
                 legend="include", name=None):
    """
    Render a categorical or continuous feature onto a tSNE/UMAP embedding.

    Parameters
    ----------
    adata : AnnData
    feature : str
        An .obs column (categorical) or a gene name in .var_names (continuous).
    var_class : str
        "categorical" or "continuous".
    reduction : str
        "umap" or "tsne" — must exist as X_<reduction> in .obsm.
    feature_colors : dict, optional
        {category: hex_color} for categorical features, or a matplotlib
        colormap name for continuous.
    target_folder : Path, optional
        If provided, saves the plot to this directory.
    file_format : str
        "png" or "pdf".
    label : bool
        If True, annotate each cluster with its label.
    legend : str
        "include", "none", or "separate".
    name : str, optional
        Object name used in the output filename. Defaults to feature.
    """
    # Select embedding
    embed_key = f"X_{reduction}"
    if embed_key not in adata.obsm:
        raise KeyError(f"{embed_key} not found in .obsm. "
                       f"Available: {list(adata.obsm.keys())}")
    coords = adata.obsm[embed_key]

    # Scale point size with cell count — matches R formula
    n = adata.n_obs
    if n > 400_000:
        pt_size = 0.001
    elif n > 100_000:
        pt_size = 0.1
    else:
        pt_size = 1.0

    fig_size = (9, 8) if legend == "include" else (8, 8)
    fig, ax = plt.subplots(figsize=fig_size)
    ax.set_aspect("equal")
    ax.axis("off")

    if var_class == "categorical":
        values = adata.obs[feature].astype(str)
        categories = values.unique()

        # Assign colors
        if feature_colors:
            palette = {str(k): v for k, v in feature_colors.items()}
        else:
            auto_colors = plt.cm.tab20.colors
            palette = {c: auto_colors[i % len(auto_colors)]
                       for i, c in enumerate(sorted(categories))}

        for cat in sorted(categories):
            mask = values == cat
            color = palette.get(cat, "#999999")
            ax.scatter(coords[mask, 0], coords[mask, 1],
                       c=color, s=pt_size, label=cat,
                       linewidths=0, rasterized=True)

        if label:
            for cat in categories:
                mask = values == cat
                cx, cy = coords[mask, 0].mean(), coords[mask, 1].mean()
                ax.text(cx, cy, cat, fontsize=8,
                        ha="center", va="center", fontweight="bold")

        if legend == "include":
            ax.legend(markerscale=max(1, 4/pt_size), fontsize=7,
                      bbox_to_anchor=(1.02, 1), loc="upper left",
                      frameon=False)

    else:  # continuous
        # Retrieve values: gene expression or obs column
        if feature in adata.var_names:
            idx = adata.var_names.get_loc(feature)
            vals = (adata.X[:, idx].toarray().flatten()
                    if sp.issparse(adata.X) else adata.X[:, idx])
        elif feature in adata.obs.columns:
            vals = adata.obs[feature].values.astype(float)
        else:
            raise KeyError(f"'{feature}' not found in .var_names or .obs")

        # Clip to q1–q9 to prevent outliers compressing the color scale
        vmin = np.percentile(vals, 10)
        vmax = np.percentile(vals, 90)

        cmap = feature_colors if isinstance(feature_colors, str) else "viridis"
        sc_plot = ax.scatter(coords[:, 0], coords[:, 1],
                             c=vals, s=pt_size, cmap=cmap,
                             vmin=vmin, vmax=vmax,
                             linewidths=0, rasterized=True)
        if legend == "include":
            plt.colorbar(sc_plot, ax=ax, shrink=0.5, label=feature)

    ax.set_title(f"{feature} ({reduction})", fontsize=10)
    plt.tight_layout()

    if target_folder is not None:
        stem = name or "object"
        out = Path(target_folder) / f"{stem}_{feature}_{reduction}.{file_format}"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"Saved: {out}")

    plt.show()

    # Separate legend file
    if legend == "separate" and var_class == "categorical":
        fig_leg, ax_leg = plt.subplots(figsize=(3, 3))
        for cat in sorted(categories):
            color = palette.get(cat, "#999999")
            ax_leg.scatter([], [], c=color, label=cat, s=40)
        ax_leg.legend(fontsize=7, frameon=False)
        ax_leg.axis("off")
        if target_folder is not None:
            leg_out = Path(target_folder) / f"{feature}_legend.pdf"
            fig_leg.savefig(leg_out, bbox_inches="tight")
            print(f"Legend saved: {leg_out}")
        plt.show()
    plt.close("all")

## Generate feature plots

The R script iterates over all four objects × both embeddings × three phenotype variables, generating 24 plots. The same loop is reproduced here.

### What each plot shows

- **`pooled_mouse_id`** — colors cells by individual animal. If the embedding is well-integrated, cells from the same type but different animals should intermix. If animals cluster separately, there is residual individual-level variation that may need to be addressed.
- **`tissue`** — confirms that the three tissues occupy distinct regions of the atlas UMAP/tSNE, as expected given their very different cell type compositions.
- **`pheno`** — colors by experimental group (SC/TC/SH/TH). Within each cell type cluster, cells from all four conditions should be present. If one condition is absent from a cluster, that cell type may be specific to a particular diet or exercise state — a biological finding worth following up.

In [ ]:
plot_features = ["pooled_mouse_id", "tissue", "pheno"]
reductions    = ["tsne", "umap"]

for df_name in df_names:
    obj_path = analysis_dir / "rdata" / f"{df_name}_combined.h5ad"
    if not obj_path.exists():
        print(f"Skipping {df_name} (not found)")
        continue

    adata = sc.read_h5ad(obj_path)
    print(f"\n{df_name}: {adata.n_obs:,} cells")

    for reduction in reductions:
        embed_key = f"X_{reduction}"
        if embed_key not in adata.obsm:
            print(f"  {embed_key} not present, skipping")
            continue

        for feat in plot_features:
            if feat not in adata.obs.columns:
                print(f"  {feat} not in .obs, skipping")
                continue

            feature_plot(
                adata=adata,
                feature=feat,
                var_class="categorical",
                reduction=reduction,
                feature_colors=annotation_colors.get(feat),
                target_folder=analysis_dir / "plots",
                file_format="png",
                label=False,
                legend="include",
                name=f"{df_name}_combined",
            )

## Additional continuous feature plots

Beyond categorical phenotype labels, it is informative to plot key marker genes and QC metrics on the embedding. This serves two purposes:

1. **Cell type verification** — plotting canonical marker genes confirms that clusters assigned to a cell type actually express the expected markers. For example, adipocyte clusters should express *Adipoq* and *Fabp4*; macrophage clusters should express *Adgre1* (F4/80) and *Csf1r*.

2. **Residual QC checks** — plotting `pct_counts_mt` on the UMAP after integration catches any clusters that are enriched for dying cells that slipped through per-sample filtering.

In [ ]:
# Canonical marker genes for major cell types in these three tissues
marker_genes = {
    "Ptprc":   "pan-immune (CD45)",
    "Hba-a1":  "red blood cells (contamination check)",
    "Adipoq":  "mature adipocytes",
    "Adgre1":  "macrophages (F4/80)",
    "Myh1":    "fast skeletal muscle fiber",
    "Myh7":    "slow skeletal muscle fiber",
    "Pdgfra":  "adipocyte precursors / FAPs",
    "Pecam1":  "endothelial cells (CD31)",
}
qc_features = ["total_counts", "n_genes_by_counts", "pct_counts_mt"]

# Load the full atlas for marker plots
atlas_path = analysis_dir / "rdata" / "scwat_vwat_skm_combined.h5ad"
if atlas_path.exists():
    adata_atlas = sc.read_h5ad(atlas_path)

    for gene, description in marker_genes.items():
        if gene not in adata_atlas.var_names:
            print(f"  {gene} not in dataset, skipping")
            continue
        print(f"\nPlotting {gene} ({description})")
        feature_plot(
            adata=adata_atlas,
            feature=gene,
            var_class="continuous",
            reduction="umap",
            target_folder=analysis_dir / "plots",
            file_format="png",
            legend="include",
            name="scwat_vwat_skm_combined",
        )

    for qc_feat in qc_features:
        feature_plot(
            adata=adata_atlas,
            feature=qc_feat,
            var_class="continuous",
            reduction="umap",
            target_folder=analysis_dir / "plots",
            file_format="png",
            legend="include",
            name="scwat_vwat_skm_combined",
        )
else:
    print(f"Atlas not found at {atlas_path} — run integration first.")